In [0]:
storage_key = dbutils.secrets.get(scope="kv-finbank", key="storage-account-key")
spark.conf.set("fs.azure.account.key.stfinbankdevfbcq2026.dfs.core.windows.net",storage_key)

In [0]:
pipeline_name = dbutils.widgets.get("pipeline_name")
activity_name = dbutils.widgets.get("activity_name")
table_name = dbutils.widgets.get("table_name")
error_message = dbutils.widgets.get("error_message")
failure_time_str = dbutils.widgets.get("failure_time")
run_id = dbutils.widgets.get("run_id")
 
print(f"registrando fallo: pipeline={pipeline_name},actividad={activity_name},tabla={table_name}")

In [0]:
from pyspark.sql import Row
from pyspark.sql.types import StructType, StructField, StringType, TimestampType
from datetime import datetime, timezone
 
LOG_ERRORES_PATH = "abfss://bronze@stfinbankdevfbcq2026.dfs.core.windows.net/_control/log_errores_pipeline"
 
schema_log_errores = StructType([
    StructField("pipeline_name", StringType(), False),
    StructField("activity_name", StringType(), False),
    StructField("table_name", StringType(), True),
    StructField("error_message", StringType(), False),
    StructField("failure_time", TimestampType(), False),
    StructField("run_id", StringType(), False),
])
try:
    failure_time_dt = datetime.strptime(failure_time_str[:19], "%Y-%m-%dT%H:%M:%S").replace(tzinfo=timezone.utc)
except Exception:
    failure_time_dt = datetime.now(timezone.utc)
 
nueva_fila = spark.createDataFrame(
    [Row(
        pipeline_name=pipeline_name,
        activity_name=activity_name,
        table_name=table_name if table_name else None,
        error_message=error_message,
        failure_time=failure_time_dt,
        run_id=run_id,
    )],
    schema=schema_log_errores
)
 
nueva_fila.write.format("delta").mode("append").save(LOG_ERRORES_PATH)
print("error registrado en log_errores_pipeline")

In [0]:
dbutils.notebook.exit("Error registrado correctamente")